# Single-class object of interest — YOLO11l-seg v3 (BCE + Dice + copy_paste)

Same recipe as `train_rtdetr_l_v2.ipynb`. Only change: **`copy_paste=0.3`**.

| Setting | L v2 | **This notebook** |
|---|---|---|
| model | `yolo11l-seg.pt` | **same** |
| mask loss | 0.5 BCE + 0.5 Dice | **same** |
| `cls` | 0.4 | **same** |
| `mosaic` / `close_mosaic` | 0.4 / 10 | **same** |
| `copy_paste` | 0.0 | **0.3** |
| `cos_lr` | True | **same** |
| `epochs` / `batch` / `imgsz` | 200 / 4 / 640 | **same** |
| `patience` | 20 | **same** |

Same single-class `train_v2/dataset`. Do **not** run the download from this notebook. Weights go to `train_v2/runs/v3/yolo11l_seg_object_bce_dice_copy_paste/` — L v2 is not overwritten.

## 0. Download the dataset (run this in a terminal first)

Skip this if `train_v2/dataset` already exists.

From the **repo root** in PowerShell:

```powershell
python train_v2/utils/download_dataset.py
```

## 1. Setup and environment check

In [1]:
from pathlib import Path
import os
import sys

import torch
import ultralytics
from ultralytics import YOLO
from ultralytics.utils import SETTINGS

SETTINGS["tensorboard"] = False

HERE = Path.cwd().resolve()
if not (HERE / "utils" / "download_dataset.py").exists():
    HERE = HERE / "train_v2"

sys.path.insert(0, str(HERE / "utils"))
from augment import LABEL_AWARE_AUG
from dice_loss import apply_bce_dice_mask_loss

DATA_YAML = HERE / "dataset" / "data.yaml"
RUNS = HERE / "runs"
L_V2_DIR = RUNS / "v2" / "yolo11l_seg_object_bce_dice"
PROJECT = RUNS / "v3"
RUN_NAME = "yolo11l_seg_object_bce_dice_copy_paste"
V3_DIR = PROJECT / RUN_NAME

os.environ["MLFLOW_EXPERIMENT_NAME"] = "train_v2_yolo11l_seg_bce_dice_copy_paste"
os.environ["MLFLOW_RUN"] = RUN_NAME

if V3_DIR.resolve() == L_V2_DIR.resolve():
    raise RuntimeError("v3 save path collided with the L v2 run.")
if V3_DIR.exists():
    print(
        f"NOTE: {V3_DIR} already exists. exist_ok=False so Ultralytics "
        "will create a new folder (…2) instead of overwriting."
    )

TRAIN_AUG = dict(LABEL_AWARE_AUG)
TRAIN_AUG["mosaic"] = 0.4
TRAIN_AUG["close_mosaic"] = 10
TRAIN_AUG["copy_paste"] = 0.3

print("=" * 70)
print("TRAIN_V2 — YOLO11l-seg v3 (BCE+Dice + copy_paste=0.3)")
print("=" * 70)
print(f"Ultralytics : {ultralytics.__version__}")
print(f"PyTorch     : {torch.__version__}")
print(f"CUDA        : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU         : {torch.cuda.get_device_name(0)}")
print()
print(f"train_v2    : {HERE}")
print(f"data.yaml   : {DATA_YAML}")
print(f"yaml exists : {DATA_YAML.exists()}")
print(f"L v2 dir    : {L_V2_DIR}  exists={L_V2_DIR.exists()}")
print(f"v3 project  : {PROJECT}")
print(f"v3 run name : {RUN_NAME}")
print(f"v3 save dir : {V3_DIR}")
print(f"MLflow exp  : {os.environ['MLFLOW_EXPERIMENT_NAME']}")
print(f"MLflow run  : {os.environ['MLFLOW_RUN']}")
if not DATA_YAML.exists():
    raise FileNotFoundError(
        "Dataset not found. From the repo root run:\n"
        "  python train_v2/utils/download_dataset.py"
    )
print("=" * 70)

c:\Users\Haqkiem\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


TRAIN_V2 — YOLO11l-seg v3 (BCE+Dice + copy_paste=0.3)
Ultralytics : 8.4.146
PyTorch     : 2.11.0+cu128
CUDA        : True
GPU         : NVIDIA GeForce RTX 4050 Laptop GPU

train_v2    : C:\Users\Haqkiem\OneDrive\UNIKL\July-2026\Competition\AIIC\PETROSAINS\train_v2
data.yaml   : C:\Users\Haqkiem\OneDrive\UNIKL\July-2026\Competition\AIIC\PETROSAINS\train_v2\dataset\data.yaml
yaml exists : True
L v2 dir    : C:\Users\Haqkiem\OneDrive\UNIKL\July-2026\Competition\AIIC\PETROSAINS\train_v2\runs\v2\yolo11l_seg_object_bce_dice  exists=True
v3 project  : C:\Users\Haqkiem\OneDrive\UNIKL\July-2026\Competition\AIIC\PETROSAINS\train_v2\runs\v3
v3 run name : yolo11l_seg_object_bce_dice_copy_paste
v3 save dir : C:\Users\Haqkiem\OneDrive\UNIKL\July-2026\Competition\AIIC\PETROSAINS\train_v2\runs\v3\yolo11l_seg_object_bce_dice_copy_paste
MLflow exp  : train_v2_yolo11l_seg_bce_dice_copy_paste
MLflow run  : yolo11l_seg_object_bce_dice_copy_paste


## 2. Confirm the local set is single-class

Every polygon should already be class `0` after the download script. This cell only checks; it does not rewrite labels.

In [2]:
import yaml

cfg = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))
print("nc   :", cfg.get("nc"))
print("names:", cfg.get("names"))

class_ids = set()
n_objects = 0
label_files = list((HERE / "dataset" / "labels").rglob("*.txt"))
for path in label_files:
    for line in path.read_text(encoding="utf-8", errors="ignore").splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        class_ids.add(int(float(parts[0])))
        n_objects += 1

print(f"label files : {len(label_files)}")
print(f"objects     : {n_objects}")
print(f"class ids   : {sorted(class_ids)}")
if class_ids != {0}:
    raise ValueError(f"Expected only class 0, found {sorted(class_ids)}")
print("OK — single class `object` (id 0). SKU labels are ignored.")

nc   : 1
names: {0: 'object'}
label files : 2726
objects     : 6067
class ids   : [0]
OK — single class `object` (id 0). SKU labels are ignored.


## 3. Switch mask loss to BCE + Dice

Patches `v8SegmentationLoss.single_mask_loss` to **0.5 BCE + 0.5 Dice** (same hybrid as L v2). Box, classification, and DFL stay Ultralytics defaults.

In [3]:
apply_bce_dice_mask_loss()

Mask loss: 0.5 BCE + 0.5 Dice (box / cls / DFL unchanged)


## 4. Augmentation (label-aware + light mosaic + copy_paste)

Same v2 augs, then **`copy_paste=0.3`**. Mosaic stays `0.4` and turns off for the last 10 epochs.

In [4]:
print("v3 train() aug kwargs:")
for key, value in TRAIN_AUG.items():
    print(f"  {key:14s} {value}")

v3 train() aug kwargs:
  mosaic         0.4
  mixup          0.0
  copy_paste     0.3
  cutmix         0.0
  perspective    0.0
  shear          0.0
  translate      0.05
  scale          0.1
  degrees        8.0
  fliplr         0.5
  flipud         0.0
  hsv_h          0.015
  hsv_s          0.5
  hsv_v          0.4
  erasing        0.0
  multi_scale    0.0
  close_mosaic   10


## 5. Train YOLO11l-seg (BCE + Dice + copy_paste, 200 epochs)

Weights land in `train_v2/runs/v3/yolo11l_seg_object_bce_dice_copy_paste/` (not the L v2 folder).

`batch=4` and `imgsz=640` are unchanged. Copy-paste uses more VRAM; drop `batch` to `2` if you OOM.

In [5]:
seg_model = YOLO("yolo11l-seg.pt")

seg_results = seg_model.train(
    data=str(DATA_YAML),
    epochs=200,
    imgsz=640,
    batch=4,
    patience=20,
    device=0 if torch.cuda.is_available() else "cpu",
    project=str(PROJECT),
    name=RUN_NAME,
    exist_ok=False,
    cls=0.4,
    cos_lr=True,
    **TRAIN_AUG,
)

print(seg_results)

c:\Users\Haqkiem\AppData\Local\Programs\Python\Python310\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


New https://pypi.org/project/ultralytics/8.4.150 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.146  Python-3.10.0 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.4, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=C:\Users\Haqkiem\OneDrive\UNIKL\July-2026\Competition\AIIC\PETROSAINS\train_v2\dataset\data.yaml, degrees=8.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.0, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.5, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=

2026/09/13 10:01:35 INFO mlflow.tracking.fluent: Experiment with name 'train_v2_yolo11l_seg_bce_dice_copy_paste' does not exist. Creating a new experiment.


MLflow: logging run_id(deb7dd3f3736467480530d355c5d5163) to runs\mlflow
MLflow: view at http://127.0.0.1:5000 with 'mlflow server --backend-store-uri runs\mlflow'
MLflow: disable with 'yolo settings mlflow=False'
Using 1909 train, 403 val images for fraction=1.0 at imgsz=640
Using 8 dataloader workers
Logging results to C:\Users\Haqkiem\OneDrive\UNIKL\July-2026\Competition\AIIC\PETROSAINS\train_v2\runs\v3\yolo11l_seg_object_bce_dice_copy_paste
Starting training for 200 epochs...

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
      1/200      4.52G      1.634      2.385      2.039      1.824          0          6        640: 100% ━━━━━━━━━━━━ 478/478 2.5it/s 3:130.5ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 51/51 1.8it/s 28.1s0.3ss
                   all        403        811       0.13       0.17     0.0579     0.0211     

error: Caught error in DataLoader worker process 3.
Original Traceback (most recent call last):
  File "c:\Users\Haqkiem\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\_utils\worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
  File "c:\Users\Haqkiem\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\_utils\fetch.py", line 54, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
  File "c:\Users\Haqkiem\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\_utils\fetch.py", line 54, in <listcomp>
    data = [self.dataset[idx] for idx in possibly_batched_index]
  File "c:\Users\Haqkiem\AppData\Local\Programs\Python\Python310\lib\site-packages\ultralytics\data\base.py", line 413, in __getitem__
    return self.transforms(self.get_image_and_label(index))
  File "c:\Users\Haqkiem\AppData\Local\Programs\Python\Python310\lib\site-packages\ultral

## 6. Quick sanity check on a val image

Loads `best.pt` from this v3 segmentation run.

In [ ]:
best = V3_DIR / "weights" / "best.pt"
if not best.exists():
    raise FileNotFoundError(f"No weights yet: {best}")

val_images = sorted(
    p for p in (HERE / "dataset" / "images" / "val").iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
)
sample = val_images[0]
print("Sample:", sample)

pred_model = YOLO(str(best))
results = pred_model.predict(source=str(sample), conf=0.25, verbose=False)[0]
print(f"detections: {len(results.boxes)}  (class ignored — all are `object`)")
out = HERE / "preview_seg_l_v3.jpg"
results.save(filename=str(out))
print("Wrote", out)